In [2]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.arima.model import ARIMA
import warnings 
from prophet import Prophet
import os

Outcome: AMD
Data: GBD 2021
Step: 1. Data preprocessing
weze_code_ver

```
Data preprocessing : AMD, SDI, SEV, RR

In [11]:
# AMD data settings (prevalence_data)
df=pd.read_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/amd2021/IHME-GBD_2021_DATA-9b3ccf22-1(rate).csv") # rate
df.head()

,measure_id,measure_name,location_id,location_name,sex_id,sex_name,age_id,age_name,cause_id,cause_name,metric_id,metric_name,year,val,upper,lower
0,5,Prevalence,9,Southeast Asia,1,Male,22,All ages,672,Age-related macular degeneration,3,Rate,1990,41.398976,50.136423,34.083862
1,5,Prevalence,9,Southeast Asia,2,Female,22,All ages,672,Age-related macular degeneration,3,Rate,1990,49.965906,60.403699,40.879057
2,5,Prevalence,9,Southeast Asia,3,Both,22,All ages,672,Age-related macular degeneration,3,Rate,1990,45.712289,54.994974,37.376211
3,5,Prevalence,9,Southeast Asia,1,Male,22,All ages,672,Age-related macular degeneration,3,Rate,1991,41.793940,50.526184,34.333852
4,5,Prevalence,9,Southeast Asia,2,Female,22,All ages,672,Age-related macular degeneration,3,Rate,1991,50.684144,61.181540,41.526245


In [12]:
# Just checking . . .
min_val = df['val'].min()
max_val = df['val'].max()
print(f"Minimum value in 'val': {min_val}")
print(f"Maximum value in 'val': {max_val}")

Minimum value in 'val': 1.2104471661
Maximum value in 'val': 4830.10906373488


In [13]:
# Log transform
df['val'] = np.log1p(df['val'])
df['upper'] = np.log1p(df['upper']) 
df['lower'] = np.log1p(df['lower'])
print(df.head()) 

   measure_id measure_name  location_id   location_name  sex_id sex_name  \
0           5   Prevalence            9  Southeast Asia       1     Male   
1           5   Prevalence            9  Southeast Asia       2   Female   
2           5   Prevalence            9  Southeast Asia       3     Both   
3           5   Prevalence            9  Southeast Asia       1     Male   
4           5   Prevalence            9  Southeast Asia       2   Female   

   age_id  age_name  cause_id                        cause_name  metric_id  \
0      22  All ages       672  Age-related macular degeneration          3   
1      22  All ages       672  Age-related macular degeneration          3   
2      22  All ages       672  Age-related macular degeneration          3   
3      22  All ages       672  Age-related macular degeneration          3   
4      22  All ages       672  Age-related macular degeneration          3   

  metric_name  year       val     upper     lower  
0        Rate  1990  3

In [14]:
amd_total = df[df['sex_id'] != 3] # filtered 'both'
amd_total_ = amd_total.drop(columns=['measure_id', 'measure_name', 'metric_id', 'metric_name', 'cause_id', 'cause_name'])
amd_total_.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/amd2021/amd_total.csv')
amd_total_.tail()

,location_id,location_name,sex_id,sex_name,age_id,age_name,year,val,upper,lower
27448,199,Western Sub-Saharan Africa,2,Female,235,95+ years,2019,7.669539,7.890602,7.424219
27450,199,Western Sub-Saharan Africa,1,Male,235,95+ years,2020,7.444806,7.689672,7.176219
27451,199,Western Sub-Saharan Africa,2,Female,235,95+ years,2020,7.670566,7.891749,7.435580
27453,199,Western Sub-Saharan Africa,1,Male,235,95+ years,2021,7.457949,7.704444,7.192730
27454,199,Western Sub-Saharan Africa,2,Female,235,95+ years,2021,7.687986,7.915130,7.455503


In [18]:
# population data setting
pop_fore=pd.read_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/population/IHME_POP_2017_2100_POP_REFERENCE_Y2020M05D01.csv") 
pop_past=pd.read_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/population/IHME_POP_2017_2100_POP_PAST_Y2020M05D01.CSV") 
min_year_fore = pop_fore['year_id'].min()
max_year_fore = pop_fore['year_id'].max()
min_year_past = pop_past['year_id'].min()
max_year_past = pop_past['year_id'].max()
print(f"Forecast Data - Min: {min_year_fore}, Max: {max_year_fore}")
print(f"Past Data - Min: {min_year_past}, Max: {max_year_past}")

Forecast Data - Min: 2018, Max: 2100
Past Data - Min: 1950, Max: 2017


In [19]:
pop_fore2 = pop_fore.drop(columns=['scenario', 'scenario_name', 'upper', 'lower'])
pop_fore2 = pop_fore2[(pop_fore2['year_id'] >= 2018) & (pop_fore2['year_id'] <= 2050)]
pop_fore2.head()

,location_id,location_name,sex_id,sex,age_group_id,age_group_name,year_id,measure_id,measure_name,metric_id,metric_name,val
0,1,Global,1,Male,2,Early Neonatal,2018,44,Population,1,Number,1368057.492
1,1,Global,1,Male,2,Early Neonatal,2019,44,Population,1,Number,1366967.702
2,1,Global,1,Male,2,Early Neonatal,2020,44,Population,1,Number,1363612.981
3,1,Global,1,Male,2,Early Neonatal,2021,44,Population,1,Number,1358904.400
4,1,Global,1,Male,2,Early Neonatal,2022,44,Population,1,Number,1354401.535


In [20]:
pop_past2 = pop_past[(pop_past['year_id'] >= 1990) & (pop_past['year_id'] <= 2017)]
pop_past2.head()

,location_id,location_name,sex_id,sex,age_group_id,age_group_name,year_id,measure_id,measure_name,metric_id,metric_name,val
40,1,Global,1,Male,2,Early Neonatal,1990,44,Population,1,Number,1365476.076
41,1,Global,1,Male,2,Early Neonatal,1991,44,Population,1,Number,1354736.723
42,1,Global,1,Male,2,Early Neonatal,1992,44,Population,1,Number,1342884.532
43,1,Global,1,Male,2,Early Neonatal,1993,44,Population,1,Number,1333022.392
44,1,Global,1,Male,2,Early Neonatal,1994,44,Population,1,Number,1327072.691


In [21]:
pop_2050 = pd.concat([pop_past2, pop_fore2], axis=0)
pop_2050 = pop_2050.rename(columns={'year_id': 'year', 'age_group_id': 'age_id', 'val': 'pop_val', 'age_group_name': 'age_name', 'sex': 'sex_name'})
pop_2050 = pop_2050.drop(columns=['location_id', 'measure_id', 'measure_name', 'metric_id', 'metric_name'])

In [22]:
print(pop_2050['age_name'].unique())

['Early Neonatal' 'Late Neonatal' 'Post Neonatal' '1 to 4' '5 to 9'
 '10 to 14' '15 to 19' '20 to 24' '25 to 29' '30 to 34' '35 to 39'
 '40 to 44' '45 to 49' '50 to 54' '55 to 59' '60 to 64' '65 to 69'
 '70 to 74' '75 to 79' 'All Ages' '80 to 84' '85 to 89' '90 to 94'
 '95 plus']


In [25]:
# age group (for new)
age_groups_to_filter = list(range(14, 22)) + [30, 31, 32, 235]
pop_2050_ = pop_2050[pop_2050['age_id'].isin(age_groups_to_filter)]
age_mapping = {
    '45 to 49': '45-49 years',
    '50 to 54': '50-54 years',
    '55 to 59': '55-59 years',
    '60 to 64': '60-64 years',
    '65 to 69': '65-69 years',
    '70 to 74': '70-74 years',
    '75 to 79': '75-79 years',
    '80 to 84': '80-84 years',
    '85 to 89': '85-89 years',
    '90 to 94': '90-94 years',
    '95 plus': '95+ years'
}
pop_2050_['age_name'] = pop_2050_['age_name'].map(age_mapping).fillna(pop_2050_['age_name'])
print(pop_2050_['age_name'].unique())

# 'age_name' 칼럼에 'All ages' 추가
all_ages = pop_2050_.groupby(['location_name', 'sex_name', 'year'], as_index=False).agg({
    'pop_val': 'sum'  # pop_val 합산
})
all_ages['age_name'] = 'All ages'
pop_2050_final = pd.concat([pop_2050_, all_ages], axis=0, ignore_index=True)
print(pop_2050_final['age_name'].unique())

# location
locations = [
    'Global', 'Australasia', 'Central Sub-Saharan Africa',
    'Southern Sub-Saharan Africa', 'East Asia', 'Oceania',
    'Southeast Asia', 'Eastern Europe', 'Central Asia',
    'North Africa and Middle East', 'Western Sub-Saharan Africa',
    'Andean Latin America', 'Southern Latin America', 'Caribbean',
    'High-income North America', 'High-income Asia Pacific',
    'Western Europe', 'Central Latin America', 'Central Europe',
    'South Asia', 'Eastern Sub-Saharan Africa',
    'Tropical Latin America'
]
pop_2050_final = pop_2050_final[pop_2050_final['location_name'].isin(locations)]
pop_2050_final = pop_2050_final.drop_duplicates(subset=['location_name', 'age_name', 'sex_name', 'year'])
pop_2050_final = pop_2050_final.drop(['sex_id', 'age_id'], axis=1)
pop_2050_final

['45-49 years' '50-54 years' '55-59 years' '60-64 years' '65-69 years'
 '70-74 years' '75-79 years' '80-84 years' '85-89 years' '90-94 years'
 '95+ years']
['45-49 years' '50-54 years' '55-59 years' '60-64 years' '65-69 years'
 '70-74 years' '75-79 years' '80-84 years' '85-89 years' '90-94 years'
 '95+ years' 'All ages']


C:\Users\psy09\AppData\Local\Temp\ipykernel_15024\247102542.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pop_2050_['age_name'] = pop_2050_['age_name'].map(age_mapping).fillna(pop_2050_['age_name'])


,location_name,sex_name,age_name,year,pop_val
0,Global,Male,45-49 years,1990,1.193935e+08
1,Global,Male,45-49 years,1991,1.210556e+08
2,Global,Male,45-49 years,1992,1.254097e+08
3,Global,Male,45-49 years,1993,1.299139e+08
4,Global,Male,45-49 years,1994,1.365640e+08
...,...,...,...,...,...
327321,Western Sub-Saharan Africa,Male,All ages,2046,7.745014e+07
327322,Western Sub-Saharan Africa,Male,All ages,2047,8.052285e+07
327323,Western Sub-Saharan Africa,Male,All ages,2048,8.376053e+07
327324,Western Sub-Saharan Africa,Male,All ages,2049,8.718568e+07


In [26]:
pop_2050_final.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/population/pop_2050.csv')
print(pop_2050_final.tail())

                     location_name sex_name  age_name  year       pop_val
327321  Western Sub-Saharan Africa     Male  All ages  2046  7.745014e+07
327322  Western Sub-Saharan Africa     Male  All ages  2047  8.052285e+07
327323  Western Sub-Saharan Africa     Male  All ages  2048  8.376053e+07
327324  Western Sub-Saharan Africa     Male  All ages  2049  8.718568e+07
327325  Western Sub-Saharan Africa     Male  All ages  2050  9.076486e+07


In [92]:
# SDI setting
sdi_df = pd.read_csv("C:/Users/psy09/Desktop/Lab/13.Adverse_effects_of_medical_treatment(GBD2021)/figure/figure_sdi_location/sdi_data/IHME_GBD_SDI_2021_SDI_1950_2021_Y2024M05D16 (2).csv")
sdi_df.head()

,covariate_name_short,location_id,location_name,year_id,age_group_id,age_group_name,sex_id,sex,mean_value,lower_value,upper_value
0,sdi,1,Global,1950,22,All Ages,3,Both,0.369235,0.369235,0.369235
1,sdi,4,"Southeast Asia, East Asia, and Oceania",1950,22,All Ages,3,Both,0.205729,0.205729,0.205729
2,sdi,5,East Asia,1950,22,All Ages,3,Both,0.193972,0.193972,0.193972
3,sdi,6,China,1950,22,All Ages,3,Both,0.184144,0.184144,0.184144
4,sdi,7,Democratic People's Republic of Korea,1950,22,All Ages,3,Both,0.323069,0.323069,0.323069


In [93]:
min_sdi_df = sdi_df['year_id'].min()
max_sdi_df = sdi_df['year_id'].max()
print(f"SDI - Min: {min_sdi_df}, Max: {max_sdi_df}")

SDI - Min: 1950, Max: 2021


In [94]:
# Prophet model
def forecast_sdi(df, target_year=2051):
    
    warnings.filterwarnings("ignore")
    forecast_df = pd.DataFrame()
    
    for (location, sex, age_group), group_data in df.groupby(['location_name', 'sex', 'age_group_name']):

        group_data = group_data.sort_values('year_id')

        df_prophet = pd.DataFrame({
            'ds': pd.to_datetime(group_data['year_id'], format='%Y'),
            'y': group_data['mean_value']  # predict_value
        })
        model = Prophet()
        model.fit(df_prophet)
        
        future = model.make_future_dataframe(periods=target_year - group_data['year_id'].max(), freq='Y')
        forecast = model.predict(future)
        forecast = forecast[forecast['ds'].dt.year <= target_year]
        forecast['location_name'] = location
        forecast['sex'] = sex
        forecast['age_group_name'] = age_group
        forecast['year_id'] = forecast['ds'].dt.year
        forecast = forecast[['location_name', 'sex', 'age_group_name', 'year_id', 'yhat']]
        forecast.rename(columns={'yhat': 'predicted_mean_value'}, inplace=True)

        forecast_df = pd.concat([forecast_df, forecast], ignore_index=True)
    
    return forecast_df

In [95]:
forecast_2050 = forecast_sdi(sdi_df, target_year=2051)

20:07:40 - cmdstanpy - INFO - Chain [1] start processing
20:07:41 - cmdstanpy - INFO - Chain [1] done processing
20:07:41 - cmdstanpy - INFO - Chain [1] start processing
20:07:41 - cmdstanpy - INFO - Chain [1] done processing
20:07:41 - cmdstanpy - INFO - Chain [1] start processing
20:07:42 - cmdstanpy - INFO - Chain [1] done processing
20:07:42 - cmdstanpy - INFO - Chain [1] start processing
20:07:42 - cmdstanpy - INFO - Chain [1] done processing
20:07:43 - cmdstanpy - INFO - Chain [1] start processing
20:07:43 - cmdstanpy - INFO - Chain [1] done processing
20:07:43 - cmdstanpy - INFO - Chain [1] start processing
20:07:43 - cmdstanpy - INFO - Chain [1] done processing
20:07:44 - cmdstanpy - INFO - Chain [1] start processing
20:07:44 - cmdstanpy - INFO - Chain [1] done processing
20:07:44 - cmdstanpy - INFO - Chain [1] start processing
20:07:45 - cmdstanpy - INFO - Chain [1] done processing
20:07:45 - cmdstanpy - INFO - Chain [1] start processing
20:07:45 - cmdstanpy - INFO - Chain [1]

In [96]:
# 'Male', 'Female' 
sdi_2050_male = forecast_2050.copy()
sdi_2050_male['sex_name'] = 'Male'
sdi_2050_male['sex_id'] = 1
sdi_2050_female = forecast_2050.copy()
sdi_2050_female['sex_name'] = 'Female'
sdi_2050_female['sex_id'] = 2
sdi_2050_updated = pd.concat([sdi_2050_male, sdi_2050_female], ignore_index=True)

print(sdi_2050_updated.tail())
print(sdi_2050_updated['sex_name'].unique())
print(sdi_2050_updated['sex_id'].unique())

       location_name   sex age_group_name  year_id  predicted_mean_value  \
149323         Ōsaka  Both       All Ages     2046              0.930728   
149324         Ōsaka  Both       All Ages     2047              0.934592   
149325         Ōsaka  Both       All Ages     2048              0.932374   
149326         Ōsaka  Both       All Ages     2049              0.935352   
149327         Ōsaka  Both       All Ages     2050              0.938754   

       sex_name  sex_id  
149323   Female       2  
149324   Female       2  
149325   Female       2  
149326   Female       2  
149327   Female       2  
['Male' 'Female']
[1 2]


In [97]:
sdi_2050 = sdi_2050_updated.rename(columns={'year_id': 'year', 'predicted_mean_value': 'sdi_val', 'age_group_name': 'age_name'})
sdi_2050 = sdi_2050.drop(columns=['sex'])
sdi_2050.tail()

,location_name,age_name,year,sdi_val,sex_name,sex_id
149323,Ōsaka,All Ages,2046,0.930728,Female,2
149324,Ōsaka,All Ages,2047,0.934592,Female,2
149325,Ōsaka,All Ages,2048,0.932374,Female,2
149326,Ōsaka,All Ages,2049,0.935352,Female,2
149327,Ōsaka,All Ages,2050,0.938754,Female,2


In [98]:
sdi_2050 = sdi_2050.rename(columns={'age_group_name': 'age_name'})
# location and age_name
locations = [
    'Global', 'Australasia', 'Central Sub-Saharan Africa',
    'Southern Sub-Saharan Africa', 'East Asia', 'Oceania',
    'Southeast Asia', 'Eastern Europe', 'Central Asia',
    'North Africa and Middle East', 'Western Sub-Saharan Africa',
    'Andean Latin America', 'Southern Latin America', 'Caribbean',
    'High-income North America', 'High-income Asia Pacific',
    'Western Europe', 'Central Latin America', 'Central Europe',
    'South Asia', 'Eastern Sub-Saharan Africa',
    'Tropical Latin America'
]
age_groups = [
    '45-49 years', '50-54 years', '55-59 years', '60-64 years',
    '65-69 years', '70-74 years', '75-79 years', '80-84 years',
    '85-89 years', '90-94 years', '95+ years'
]

filtered_sdi_2050 = sdi_2050[sdi_2050['location_name'].isin(locations)]
for age in age_groups:
    new_row = filtered_sdi_2050.copy()
    new_row['age_name'] = age
    filtered_sdi_2050 = pd.concat([filtered_sdi_2050, new_row], ignore_index=True)  
print(filtered_sdi_2050)

                      location_name   age_name  year   sdi_val sex_name  \
0              Andean Latin America   All Ages  1950  0.290058     Male   
1              Andean Latin America   All Ages  1951  0.293404     Male   
2              Andean Latin America   All Ages  1952  0.296573     Male   
3              Andean Latin America   All Ages  1953  0.299706     Male   
4              Andean Latin America   All Ages  1954  0.303216     Male   
...                             ...        ...   ...       ...      ...   
9191419  Western Sub-Saharan Africa  95+ years  2046  0.626398   Female   
9191420  Western Sub-Saharan Africa  95+ years  2047  0.632963   Female   
9191421  Western Sub-Saharan Africa  95+ years  2048  0.640824   Female   
9191422  Western Sub-Saharan Africa  95+ years  2049  0.647495   Female   
9191423  Western Sub-Saharan Africa  95+ years  2050  0.654107   Female   

         sex_id  
0             1  
1             1  
2             1  
3             1  
4        

In [107]:
# 'All Ages'를 'All ages'로 변경
filtered_sdi_2050['age_name'] = filtered_sdi_2050['age_name'].replace('All Ages', 'All ages')
filtered_sdi_2050 = filtered_sdi_2050[filtered_sdi_2050['year'] >= 1990]  

print(filtered_sdi_2050['age_name'].unique())
print(filtered_sdi_2050.tail())
file_path = "C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/SDI(forecast).csv"
filtered_sdi_2050.to_csv(file_path, index=False)

['All ages' '45-49 years' '50-54 years' '55-59 years' '60-64 years'
 '65-69 years' '70-74 years' '75-79 years' '80-84 years' '85-89 years'
 '90-94 years' '95+ years']
                      location_name   age_name  year   sdi_val sex_name  \
9191419  Western Sub-Saharan Africa  95+ years  2046  0.626398   Female   
9191420  Western Sub-Saharan Africa  95+ years  2047  0.632963   Female   
9191421  Western Sub-Saharan Africa  95+ years  2048  0.640824   Female   
9191422  Western Sub-Saharan Africa  95+ years  2049  0.647495   Female   
9191423  Western Sub-Saharan Africa  95+ years  2050  0.654107   Female   

         sex_id  
9191419       2  
9191420       2  
9191421       2  
9191422       2  
9191423       2  


In [12]:
# SEV
sev = pd.read_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/AMD_SEV.csv") 
sev.head()

,measure_id,measure_name,location_id,location_name,sex_id,sex_name,age_id,age_name,rei_id,rei_name,metric_id,metric_name,year,val,upper,lower
0,29,Summary exposure value,1,Global,1,Male,14,45-49 years,99,Smoking,3,Rate,1990,40.196120,41.546096,39.015978
1,29,Summary exposure value,1,Global,2,Female,14,45-49 years,99,Smoking,3,Rate,1990,11.225829,11.882613,10.676591
2,29,Summary exposure value,1,Global,1,Male,15,50-54 years,99,Smoking,3,Rate,1990,40.758635,42.194502,39.483607
3,29,Summary exposure value,1,Global,2,Female,15,50-54 years,99,Smoking,3,Rate,1990,10.814782,11.433870,10.206250
4,29,Summary exposure value,1,Global,1,Male,16,55-59 years,99,Smoking,3,Rate,1990,38.383574,39.824604,37.031484


In [15]:
sev['location_name'].unique()

array(['Global', 'Australasia', 'Central Sub-Saharan Africa',
       'Southern Sub-Saharan Africa', 'East Asia', 'Oceania',
       'Southeast Asia', 'Eastern Europe', 'Central Asia',
       'North Africa and Middle East', 'Western Sub-Saharan Africa',
       'Andean Latin America', 'Southern Latin America', 'Caribbean',
       'High-income North America', 'High-income Asia Pacific',
       'Western Europe', 'Central Latin America', 'Central Europe',
       'South Asia', 'Eastern Sub-Saharan Africa',
       'Tropical Latin America'], dtype=object)

In [61]:
def forecast_sev(sev_data, end_year=2051):
    sev_forecast_all = pd.DataFrame()

    # location_name, sex_id, age_name
    for location in sev_data['location_name'].unique():
        for sex in sev_data['sex_id'].unique():
            for age in sev_data['age_name'].unique():
                sev_subset = sev_data[
                    (sev_data['location_name'] == location) &
                    (sev_data['sex_id'] == sex) &
                    (sev_data['age_name'] == age)
                ]

                if sev_subset.empty:
                    continue

                sev_subset = sev_subset.rename(columns={'year': 'ds', 'val': 'y'})[['ds', 'y', 'age_id', 'sex_name', 'upper', 'lower']]
                sev_subset['ds'] = pd.to_datetime(sev_subset['ds'], format='%Y')

                # Prophet
                model = Prophet()
                model.fit(sev_subset[['ds', 'y']])
                future = model.make_future_dataframe(periods=end_year - sev_subset['ds'].dt.year.max(), freq='Y')
                forecast = model.predict(future)
                forecast = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].rename(
                    columns={'ds': 'year', 'yhat': 'val', 'yhat_lower': 'lower', 'yhat_upper': 'upper'}
                )
                forecast['year'] = forecast['year'].dt.year
                forecast['location_name'] = location
                forecast['sex_id'] = sex
                forecast['age_name'] = age
                forecast['age_id'] = sev_subset['age_id'].iloc[0] 
                forecast['sex_name'] = sev_subset['sex_name'].iloc[0] 
                
                sev_forecast_all = pd.concat([sev_forecast_all, forecast])

    return sev_forecast_all.reset_index(drop=True)

# run (forcast)
sev_forecast = forecast_sev(sev)

22:41:44 - cmdstanpy - INFO - Chain [1] start processing
22:41:44 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\psy09\AppData\Local\Programs\Python\Python312\Lib\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = pd.date_range(
22:41:44 - cmdstanpy - INFO - Chain [1] start processing
22:41:45 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\psy09\AppData\Local\Programs\Python\Python312\Lib\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = pd.date_range(
22:41:45 - cmdstanpy - INFO - Chain [1] start processing
22:41:45 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\psy09\AppData\Local\Programs\Python\Python312\Lib\site-packages\prophet\forecaster.py:1854: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = p

In [63]:
sev_forecast.to_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/AMD_SEV(forecast).csv")
print(sev_forecast.tail())

       year       val     lower     upper           location_name  sex_id  \
32731  2046  1.118779 -2.124605  4.366699  Tropical Latin America       2   
32732  2047  1.022674 -2.381158  4.478648  Tropical Latin America       2   
32733  2048  0.968613 -2.601183  4.564697  Tropical Latin America       2   
32734  2049  0.873432 -2.875856  4.657567  Tropical Latin America       2   
32735  2050  0.777619 -3.196680  4.765968  Tropical Latin America       2   

        age_name  age_id sex_name  
32731  95+ years     235   Female  
32732  95+ years     235   Female  
32733  95+ years     235   Female  
32734  95+ years     235   Female  
32735  95+ years     235   Female  


In [89]:
sev_forecast=pd.read_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/AMD_SEV(forecast).csv")
sev_forecast = sev_forecast.loc[:, ~sev_forecast.columns.str.contains('^Unnamed')]

smoke_categories = [
    '0 Cigarettes Per Day',
    '10 Cigarettes Per Day',
    '20 Cigarettes Per Day',
    '30 Cigarettes Per Day',
    '40 Cigarettes Per Day',
    '50 Cigarettes Per Day',
    '60 Cigarettes Per Day',
    'Mean Per Day'
]

sev_expanded = pd.DataFrame()
for smoke_cate in smoke_categories:
    temp = sev_forecast.copy()
    temp['smoke_cate'] = smoke_cate
    sev_expanded = pd.concat([sev_expanded, temp], axis=0)

print(sev_expanded.head(16)) 
sev_expanded.to_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/AMD_SEV(forecast).csv")

    year        val      lower      upper location_name  sex_id     age_name  \
0   1990  40.182944  40.144013  40.221123        Global       1  45-49 years   
1   1991  39.908633  39.871321  39.943873        Global       1  45-49 years   
2   1992  39.738234  39.698750  39.774672        Global       1  45-49 years   
3   1993  39.620800  39.583497  39.658905        Global       1  45-49 years   
4   1994  39.617769  39.581171  39.656442        Global       1  45-49 years   
5   1995  39.552259  39.512347  39.591244        Global       1  45-49 years   
6   1996  39.321952  39.285177  39.360830        Global       1  45-49 years   
7   1997  39.003943  38.965488  39.040599        Global       1  45-49 years   
8   1998  38.683192  38.648543  38.720775        Global       1  45-49 years   
9   1999  38.344323  38.304514  38.381523        Global       1  45-49 years   
10  2000  38.029913  37.995455  38.066295        Global       1  45-49 years   
11  2001  37.745252  37.703739  37.78320

In [59]:
# RR
rr = pd.read_excel("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/AMD_RR(smoking).XLSX") 
rr.head()

,Risk,Category / Units,Sex,45-49 years,50-54 years,55-59 years,60-64 years,65-69 years,70-74 years,75-79 years,80-84 years,85-89 years,90-94 years,95+ years
0,Age-related macular degeneration,0 Cigarettes Per Day,Female,1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00)
1,Age-related macular degeneration,0 Cigarettes Per Day,Male,1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00),1.00 \n(1.00 to 1.00)
2,Age-related macular degeneration,10 Cigarettes Per Day,Female,1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88)
3,Age-related macular degeneration,10 Cigarettes Per Day,Male,1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88),1.45 \n(1.09 to 1.88)
4,Age-related macular degeneration,20 Cigarettes Per Day,Female,1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48),1.91 \n(1.39 to 2.48)


In [60]:
rr1 = rr.drop(columns=['Risk'])
rr2 = rr1.rename(columns={'Category / Units': 'smoke_cate', 'Sex' : 'sex_name'})
rr2['sex_id'] = rr2['sex_name'].apply(lambda x: 2 if x == 'Female' else 1)
# \n
columns_to_clean = [col for col in rr2.columns if "years" in col]
rr2[columns_to_clean] = rr2[columns_to_clean].apply(lambda col: col.str.replace("\n", " "))
print(rr2.head())

              smoke_cate sex_name           45-49 years           50-54 years  \
0   0 Cigarettes Per Day   Female  1.00  (1.00 to 1.00)  1.00  (1.00 to 1.00)   
1   0 Cigarettes Per Day     Male  1.00  (1.00 to 1.00)  1.00  (1.00 to 1.00)   
2  10 Cigarettes Per Day   Female  1.45  (1.09 to 1.88)  1.45  (1.09 to 1.88)   
3  10 Cigarettes Per Day     Male  1.45  (1.09 to 1.88)  1.45  (1.09 to 1.88)   
4  20 Cigarettes Per Day   Female  1.91  (1.39 to 2.48)  1.91  (1.39 to 2.48)   

            55-59 years           60-64 years           65-69 years  \
0  1.00  (1.00 to 1.00)  1.00  (1.00 to 1.00)  1.00  (1.00 to 1.00)   
1  1.00  (1.00 to 1.00)  1.00  (1.00 to 1.00)  1.00  (1.00 to 1.00)   
2  1.45  (1.09 to 1.88)  1.45  (1.09 to 1.88)  1.45  (1.09 to 1.88)   
3  1.45  (1.09 to 1.88)  1.45  (1.09 to 1.88)  1.45  (1.09 to 1.88)   
4  1.91  (1.39 to 2.48)  1.91  (1.39 to 2.48)  1.91  (1.39 to 2.48)   

            70-74 years           75-79 years           80-84 years  \
0  1.00  (1.00 

In [61]:
rr_melted = rr2.melt(
    id_vars=['smoke_cate', 'sex_id', 'sex_name'], 
    var_name='age_name', 
    value_name='RR'
)
print(rr_melted.tail())

                smoke_cate  sex_id sex_name   age_name                    RR
149  40 Cigarettes Per Day       1     Male  95+ years  2.70  (1.70 to 3.89)
150  50 Cigarettes Per Day       2   Female  95+ years  2.89  (1.58 to 4.65)
151  50 Cigarettes Per Day       1     Male  95+ years  2.89  (1.58 to 4.65)
152  60 Cigarettes Per Day       2   Female  95+ years  3.08  (1.46 to 5.42)
153  60 Cigarettes Per Day       1     Male  95+ years  3.08  (1.46 to 5.42)


In [62]:
# 95% CI
import re
def split_rr(row):
    rr_value = re.search(r"^\d+\.?\d*", row).group() if re.search(r"^\d+\.?\d*", row) else None
    range_match = re.search(r"\(([\d.]+) to ([\d.]+)\)", row)
    lower_value = range_match.group(1) if range_match else None
    upper_value = range_match.group(2) if range_match else None
    return rr_value, lower_value, upper_value

rr_melted[['RR', 'RR_lower', 'RR_upper']] = rr_melted['RR'].apply(lambda x: pd.Series(split_rr(x)))
print(rr_melted.tail())

                smoke_cate  sex_id sex_name   age_name    RR RR_lower RR_upper
149  40 Cigarettes Per Day       1     Male  95+ years  2.70     1.70     3.89
150  50 Cigarettes Per Day       2   Female  95+ years  2.89     1.58     4.65
151  50 Cigarettes Per Day       1     Male  95+ years  2.89     1.58     4.65
152  60 Cigarettes Per Day       2   Female  95+ years  3.08     1.46     5.42
153  60 Cigarettes Per Day       1     Male  95+ years  3.08     1.46     5.42


In [63]:
rr_melted['RR'] = pd.to_numeric(rr_melted['RR'], errors='coerce')
rr_melted['RR_lower'] = pd.to_numeric(rr_melted['RR_lower'], errors='coerce')
rr_melted['RR_upper'] = pd.to_numeric(rr_melted['RR_upper'], errors='coerce')
rr_melted['smoke_cate'].unique()

array(['0 Cigarettes Per Day', '10 Cigarettes Per Day',
       '20 Cigarettes Per Day', '30 Cigarettes Per Day',
       '40 Cigarettes Per Day', '50 Cigarettes Per Day',
       '60 Cigarettes Per Day'], dtype=object)

In [64]:
# smoke_cate : mean cigarattes
mean_per_day = rr_melted.groupby(['sex_id', 'age_name', 'sex_name'])[['RR', 'RR_lower', 'RR_upper']].mean().reset_index()
mean_per_day['smoke_cate'] = 'Mean Per Day'
rr_final = pd.concat([rr_melted, mean_per_day], ignore_index=True)
print(rr_final.tail())

       smoke_cate  sex_id sex_name     age_name        RR  RR_lower  RR_upper
171  Mean Per Day       2   Female  75-79 years  2.207143  1.418571  3.208571
172  Mean Per Day       2   Female  80-84 years  2.207143  1.418571  3.208571
173  Mean Per Day       2   Female  85-89 years  2.207143  1.418571  3.208571
174  Mean Per Day       2   Female  90-94 years  2.207143  1.418571  3.208571
175  Mean Per Day       2   Female    95+ years  2.207143  1.418571  3.208571


In [65]:
rr_final['age_name'].unique()

array(['45-49 years', '50-54 years', '55-59 years', '60-64 years',
       '65-69 years', '70-74 years', '75-79 years', '80-84 years',
       '85-89 years', '90-94 years', '95+ years'], dtype=object)

In [73]:
# all age
all_age = rr_final.groupby(['smoke_cate', 'sex_name', 'sex_id'])[['RR', 'RR_lower', 'RR_upper']].mean().reset_index()
all_age['age_name'] = 'All ages'
rr_final = pd.concat([rr_final, all_age], ignore_index=True)

In [78]:
# location
locations = [
    'Global', 'Australasia', 'Central Sub-Saharan Africa',
    'Southern Sub-Saharan Africa', 'East Asia', 'Oceania',
    'Southeast Asia', 'Eastern Europe', 'Central Asia',
    'North Africa and Middle East', 'Western Sub-Saharan Africa',
    'Andean Latin America', 'Southern Latin America', 'Caribbean',
    'High-income North America', 'High-income Asia Pacific',
    'Western Europe', 'Central Latin America', 'Central Europe',
    'South Asia', 'Eastern Sub-Saharan Africa',
    'Tropical Latin America'
]
expanded_data = []
for location in locations:
    temp_df = rr_final.copy()
    temp_df['location_name'] = location
    expanded_data.append(temp_df)
rr_expanded = pd.concat(expanded_data, ignore_index=True)

print(rr_expanded['age_name'].unique()) 
print(rr_expanded['location_name'].unique())

['45-49 years' '50-54 years' '55-59 years' '60-64 years' '65-69 years'
 '70-74 years' '75-79 years' '80-84 years' '85-89 years' '90-94 years'
 '95+ years' 'All ages']
['Global' 'Australasia' 'Central Sub-Saharan Africa'
 'Southern Sub-Saharan Africa' 'East Asia' 'Oceania' 'Southeast Asia'
 'Eastern Europe' 'Central Asia' 'North Africa and Middle East'
 'Western Sub-Saharan Africa' 'Andean Latin America'
 'Southern Latin America' 'Caribbean' 'High-income North America'
 'High-income Asia Pacific' 'Western Europe' 'Central Latin America'
 'Central Europe' 'South Asia' 'Eastern Sub-Saharan Africa'
 'Tropical Latin America']


In [81]:
years = np.arange(1990, 2051)

def expand_rr_with_all_columns(rr_expanded, years):
    expanded_rr = pd.DataFrame()
    
    for year in years:
        temp = rr_expanded.copy()  
        temp['year'] = year 
        
        expanded_rr = pd.concat([expanded_rr, temp], axis=0)
    
    return expanded_rr

rr_with_years = expand_rr_with_all_columns(rr_expanded, years)

unique_combinations = rr_with_years.drop_duplicates(subset=['location_name', 'smoke_cate', 'sex_name', 'age_name', 'year'])
unique_combinations.to_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/AMD_RR(final).csv", index=False)
unique_combinations


,smoke_cate,sex_id,sex_name,age_name,RR,RR_lower,RR_upper,location_name,year
0,0 Cigarettes Per Day,2,Female,45-49 years,1.000000,1.000000,1.000000,Global,1990
1,0 Cigarettes Per Day,1,Male,45-49 years,1.000000,1.000000,1.000000,Global,1990
2,10 Cigarettes Per Day,2,Female,45-49 years,1.450000,1.090000,1.880000,Global,1990
3,10 Cigarettes Per Day,1,Male,45-49 years,1.450000,1.090000,1.880000,Global,1990
4,20 Cigarettes Per Day,2,Female,45-49 years,1.910000,1.390000,2.480000,Global,1990
...,...,...,...,...,...,...,...,...,...
4555,50 Cigarettes Per Day,1,Male,All ages,2.890000,1.580000,4.650000,Tropical Latin America,2050
4556,60 Cigarettes Per Day,2,Female,All ages,3.080000,1.460000,5.420000,Tropical Latin America,2050
4557,60 Cigarettes Per Day,1,Male,All ages,3.080000,1.460000,5.420000,Tropical Latin America,2050
4558,Mean Per Day,2,Female,All ages,2.207143,1.418571,3.208571,Tropical Latin America,2050
